# 📊 Portfolio News Search - Bigdata.com API Demo

This notebook demonstrates a complete workflow for searching financial news across a portfolio of tickers using the Bigdata.com API.

## Features
- **Entity Resolution**: Lookup tickers via Knowledge Graph API with CSV caching
- **Parallel Processing**: ThreadPoolExecutor for high-performance searches
- **Rate Limiting**: Token bucket algorithm (500 requests/min)
- **SQLite Storage**: Persistent storage for search results
- **Query Interface**: Filter by ticker or topic

## Workflow
1. Configure inputs (tickers, dates, topics)
2. Resolve tickers to entity IDs (with caching)
3. Execute parallel topic searches
4. Store results in SQLite database
5. Query results by ticker or topic

---


In [ ]:
# 📦 Install required dependencies
!pip install requests python-dotenv httpx

## 1️⃣ Configuration & User Inputs

**Modify the values below to customize your search:**


In [ ]:
# =============================================================================
# 📝 USER INPUTS - MODIFY THESE VALUES
# =============================================================================

# Smaller Set: for test 
#TICKERS_INPUT = "AAPL, MSFT, GOOGL, TSLA, NVDA"

TICKERS_INPUT = "AAPL,ABNB,ADBE,ADI,ADP,ADSK,AEP,ALNY,AMAT,AMD,AMGN,AMZN,ANSS,APP,ARM,ASML,AVGO,AXON,AZN,BKR,BKNG,CDNS,CEG,CHTR,CMCSA,COST,CPRT,CRWD,CSCO,CSGP,CSX,CTAS,CTSH,DASH,DDOG,DXCM,EA,EXC,FANG,FAST,FER,FTNT,GEHC,GILD,GOOGL,HON,IDXX,INTC,INTU,ISRG,KDP,KHC,KLAC,LRCX,LIN,MAR,MCHP,MDLZ,MELI,META,MNST,MPWR,MSFT,MSTR,MU,NFLX,NVDA,NXPI,ODFL,ORLY,PANW,PAYX,PCAR,PDD,PEP,PLTR,PYPL,QCOM,REGN,ROP,ROST,SBUX,SHOP,SNPS,STX,TEAM,TMUS,TSLA,TTWO,TXN,VRSK,VRTX,VSNT,WBD,WDAY,WDC,XEL,ZS"


# Date range for search (YYYY-MM-DD format)
START_DATE = "2025-01-01"
END_DATE = "2025-01-12"

# Search topics - customize these for your research needs
# Use {company} placeholder for company name substitution
TOPICS = [
    # Earnings & Financial Performance
    {"topic_name": "Financial Metrics", "topic_text": "What key takeaways emerged from {company}'s latest earnings report?"},
    {"topic_name": "Financial Metrics", "topic_text": "What notable changes in {company}'s financial performance metrics have been reported recently?"},
    {"topic_name": "Financial Metrics", "topic_text": "Has {company} revised its financial or operational guidance for upcoming periods?"},
    
    # Strategy & Business Development
    {"topic_name": "M&A", "topic_text": "What significant strategic initiatives or business pivots has {company} announced recently?"},
    {"topic_name": "M&A", "topic_text": "What material acquisition, merger, or divestiture activities involve {company} currently?"},
    
    # Leadership & Organization
    {"topic_name": "Leadership", "topic_text": "What executive leadership changes have been announced at {company} recently?"},
    
    # Commercial & Market Activity
    {"topic_name": "Competition", "topic_text": "What significant contract wins, losses, or renewals has {company} recently announced?"},
    {"topic_name": "Competition", "topic_text": "What notable market share shifts has {company} experienced recently?"},
    {"topic_name": "Competition", "topic_text": "How is {company} responding to new competitive threats or significant competitor actions?"},
    
    # Product & Innovation
    {"topic_name": "Products", "topic_text": "What significant new product launches or pipeline developments has {company} announced?"},
    
    # Operations & Supply Chain
    {"topic_name": "Supply Chain", "topic_text": "What material operational disruptions or capacity changes is {company} experiencing currently?"},
    {"topic_name": "Supply Chain", "topic_text": "How are supply chain conditions affecting {company}'s operations and outlook?"},
    {"topic_name": "Supply Chain", "topic_text": "What production milestones or efficiency improvements has {company} achieved recently?"},
    
    # Cost Management
    {"topic_name": "Costs", "topic_text": "What cost-cutting measures or expense management initiatives has {company} recently disclosed?"},
    
    # Regulatory & Legal
    {"topic_name": "Regulatory", "topic_text": "What specific regulatory developments are materially affecting {company}?"},
    {"topic_name": "Regulatory", "topic_text": "What material litigation developments involve {company} currently?"},
    
    # Macro & Industry Trends
    {"topic_name": "Industry", "topic_text": "How are current macroeconomic factors affecting {company}'s performance and outlook?"},
    {"topic_name": "Industry", "topic_text": "What industry-specific trends or disruptions are directly affecting {company}?"},
    
    # Capital Allocation & Financing
    {"topic_name": "Financing", "topic_text": "What significant capital allocation decisions has {company} announced recently?"},
    {"topic_name": "Financing", "topic_text": "What changes to dividends, buybacks, or other shareholder return programs has {company} announced?"},
    {"topic_name": "Financing", "topic_text": "What debt issuance, refinancing, or covenant changes has {company} recently announced?"},
    {"topic_name": "Financing", "topic_text": "Have there been any credit rating actions or outlook changes for {company} recently?"},
    
    # Market Sentiment & Events
    {"topic_name": "Markets", "topic_text": "What shifts in the prevailing narrative around {company} are emerging among influential investors?"},
    {"topic_name": "Markets", "topic_text": "What significant events could impact {company}'s performance in the near term?"},
    {"topic_name": "Markets", "topic_text": "What unexpected disclosures or unusual trading patterns has {company} experienced recently?"},
    {"topic_name": "Markets", "topic_text": "Is there any activist investor involvement or significant shareholder actions affecting {company}?"},
]

# =============================================================================
# 🔧 ADVANCED SETTINGS (modify if needed)
# =============================================================================

# Rate limiting: 500 requests per minute max (API limit)
# Using 460 with 8% safety margin and sliding window approach
MAX_REQUESTS_PER_MINUTE = 460

# Max chunks per search query (higher = more results but slower)
MAX_CHUNKS_PER_QUERY = 10

# Document types to include
DOCUMENT_TYPES = ["NEWS", "TRANSCRIPT"]

# Sentiment filter (positive/negative only for more relevant results)
SENTIMENT_VALUES = ["positive", "negative"]

# Parallel workers for entity resolution
ENTITY_WORKERS = 10

# Parallel workers for search - limits simultaneous connections
SEARCH_WORKERS = 10

# Sliding window parameters for burst prevention
WINDOW_SIZE_SECONDS = 5
MAX_RETRIES = 5  # sufficient with multi-layered protection

print("✅ Configuration loaded")


✅ Configuration loaded


## 2️⃣ Setup & Dependencies


In [3]:
import os
import time
import json
import sqlite3
import threading
import csv
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Tuple
from pathlib import Path
from collections import defaultdict

import requests
http_session = requests.Session()
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# =============================================================================
# VALIDATE API KEY
# =============================================================================
API_KEY = os.getenv("BIGDATA_API_KEY")
if not API_KEY:
    raise ValueError(
        "❌ BIGDATA_API_KEY not found!\n"
        "Please create a .env file with:\n"
        "BIGDATA_API_KEY=your_api_key_here"
    )

# =============================================================================
# CONSTANTS & PATHS
# =============================================================================
BASE_URL = "https://api.bigdata.com/v1"
OUTPUT_DIR_NAME = "output"
OUTPUT_DIR = Path(OUTPUT_DIR_NAME)
OUTPUT_DIR.mkdir(exist_ok=True)

# Cache file for entity lookups (persists between runs)
ENTITY_CACHE_FILE = OUTPUT_DIR / "entity_cache.csv"

# SQLite database for search results
DATABASE_FILE = OUTPUT_DIR / "search_results.db"

# Parse tickers from input string OR use entity IDs from CSV
if TICKERS_INPUT:
    TICKERS = [t.strip().upper() for t in TICKERS_INPUT.split(",") if t.strip()]
    USE_ENTITY_IDS_DIRECTLY = False
else:
    TICKERS = []  # Will use ENTITY_IDS_FROM_CSV instead
    USE_ENTITY_IDS_DIRECTLY = True

print("✅ Dependencies loaded successfully")
print(f"✅ API Key configured: {API_KEY[:8]}...{API_KEY[-4:]}")
print(f"\n📁 Output directory: {OUTPUT_DIR_NAME}")
if USE_ENTITY_IDS_DIRECTLY:
    print(f"📊 Using {len(ENTITY_IDS_FROM_CSV)} entity IDs from CSV")
else:
    print(f"📊 Tickers to search: {TICKERS}")
print(f"📅 Date range: {START_DATE} to {END_DATE}")
print(f"📝 Topics configured: {len(TOPICS)}")


✅ Dependencies loaded successfully
✅ API Key configured: bd_v1_NW...2252

📁 Output directory: output
📊 Tickers to search: ['AAPL', 'ABNB', 'ADBE', 'ADI', 'ADP', 'ADSK', 'AEP', 'ALNY', 'AMAT', 'AMD', 'AMGN', 'AMZN', 'ANSS', 'APP', 'ARM', 'ASML', 'AVGO', 'AXON', 'AZN', 'BKR', 'BKNG', 'CDNS', 'CEG', 'CHTR', 'CMCSA', 'COST', 'CPRT', 'CRWD', 'CSCO', 'CSGP', 'CSX', 'CTAS', 'CTSH', 'DASH', 'DDOG', 'DXCM', 'EA', 'EXC', 'FANG', 'FAST', 'FER', 'FTNT', 'GEHC', 'GILD', 'GOOGL', 'HON', 'IDXX', 'INTC', 'INTU', 'ISRG', 'KDP', 'KHC', 'KLAC', 'LRCX', 'LIN', 'MAR', 'MCHP', 'MDLZ', 'MELI', 'META', 'MNST', 'MPWR', 'MSFT', 'MSTR', 'MU', 'NFLX', 'NVDA', 'NXPI', 'ODFL', 'ORLY', 'PANW', 'PAYX', 'PCAR', 'PDD', 'PEP', 'PLTR', 'PYPL', 'QCOM', 'REGN', 'ROP', 'ROST', 'SBUX', 'SHOP', 'SNPS', 'STX', 'TEAM', 'TMUS', 'TSLA', 'TTWO', 'TXN', 'VRSK', 'VRTX', 'VSNT', 'WBD', 'WDAY', 'WDC', 'XEL', 'ZS']
📅 Date range: 2025-01-01 to 2025-01-12
📝 Topics configured: 26


## 3️⃣ Multi-Layered Rate Limiting & Protection

Robust three-layer approach to prevent API rate limiting and security filter triggers:

### 🛡️ Layer 1: Sliding Window Rate Limiter
- **460 requests/min** (8% safety margin from 500 req/min API limit)
- **5-second windows** to prevent request bursts that trigger WAF
- Smoothly distributes requests over time
- Tracks requests in sliding windows to avoid concentrated spikes

### 🔒 Layer 2: Concurrency Semaphore
- **Max 40 simultaneous connections**
- Prevents connection spikes that can trigger security filters
- Controls thread pool to avoid overwhelming the API endpoint

### 🔄 Layer 3: Automatic Retry Logic
- **Up to 5 retry attempts** with intelligent backoff (sufficient with multi-layered protection)
- **1-second wait** on 429 (rate limit) errors, automatic retry
- **Progressive backoff** on 403 (WAF) errors
- **Warning every 100 retries** to track persistent issues
- Gracefully handles transient network issues

**Key Insight:** The high throttle events and wait time you see are spread across 40 parallel threads, so the actual wall-clock time is much shorter than the cumulative wait time.


In [4]:
from collections import deque

class SlidingWindowRateLimiter:
    """
    Thread-safe sliding window rate limiter with burst prevention.
    Uses 5-second windows to prevent request bursts.
    """
    
    def __init__(self, max_requests: int = 460, period_seconds: int = 60, window_size: int = 5):
        """
        Initialize sliding window rate limiter.
        
        Args:
            max_requests: Maximum requests allowed per period (default: 460 with 8% safety margin)
            period_seconds: Time period in seconds (default: 60 for per-minute)
            window_size: Size of sliding window in seconds (default: 5 for burst prevention)
        """
        self.max_requests = max_requests
        self.period_seconds = period_seconds
        self.window_size = window_size
        self.max_per_window = int(max_requests * window_size / period_seconds)
        
        # Track requests in sliding windows
        self.request_times = deque()
        self._lock = threading.Lock()
        
        # Metrics tracking
        self.total_requests = 0
        self.total_wait_time = 0.0
        self.throttle_events = 0
        self.rate_limit_warnings = 0
    
    def _clean_old_requests(self, current_time: float) -> None:
        """Remove requests outside the sliding window."""
        cutoff_time = current_time - self.period_seconds
        while self.request_times and self.request_times[0] < cutoff_time:
            self.request_times.popleft()
    
    def _requests_in_window(self, current_time: float) -> int:
        """Count requests in the current window."""
        window_start = current_time - self.window_size
        return sum(1 for t in self.request_times if t >= window_start)
    
    def acquire(self, timeout: float = 60.0) -> float:
        """
        Acquire permission to make a request, blocking if necessary.
        
        Args:
            timeout: Maximum time to wait in seconds
            
        Returns:
            Wait time in seconds (0 if no wait was needed)
        """
        start_time = time.time()
        total_wait = 0.0
        
        while True:
            with self._lock:
                current_time = time.time()
                self._clean_old_requests(current_time)
                
                # Check if we can make a request
                requests_in_period = len(self.request_times)
                requests_in_window = self._requests_in_window(current_time)
                
                if requests_in_period < self.max_requests and requests_in_window < self.max_per_window:
                    # Permission granted
                    self.request_times.append(current_time)
                    self.total_requests += 1
                    self.total_wait_time += total_wait
                    return total_wait
                
                # Need to wait
                self.throttle_events += 1
                if requests_in_window >= self.max_per_window:
                    # Wait for window to clear
                    wait_time = self.window_size / 10  # Small incremental wait
                else:
                    # Wait for period to clear
                    oldest_request = self.request_times[0]
                    wait_time = (oldest_request + self.period_seconds - current_time) + 0.1
            
            # Check timeout
            if time.time() - start_time > timeout:
                raise TimeoutError("Rate limiter timeout exceeded")
            
            # Wait outside the lock
            time.sleep(min(wait_time, 1.0))
            total_wait += wait_time
    
    def get_stats(self) -> dict:
        """Get rate limiter statistics."""
        with self._lock:
            return {
                "total_requests": self.total_requests,
                "total_wait_time_seconds": round(self.total_wait_time, 2),
                "throttle_events": self.throttle_events,
                "rate_limit_warnings": self.rate_limit_warnings,
                "current_requests_in_period": len(self.request_times),
                "max_requests_per_period": self.max_requests,
                "max_requests_per_window": self.max_per_window,
            }


class ConcurrencySemaphore:
    """
    Semaphore to limit simultaneous connections.
    Prevents connection spikes that can trigger security filters.
    """
    
    def __init__(self, max_concurrent: int = 40):
        """
        Initialize concurrency semaphore.
        
        Args:
            max_concurrent: Maximum simultaneous connections (default: 40)
        """
        self.semaphore = threading.Semaphore(max_concurrent)
        self.max_concurrent = max_concurrent
        self.active_count = 0
        self._lock = threading.Lock()
        
        # Metrics
        self.total_acquisitions = 0
        self.peak_concurrent = 0
    
    def __enter__(self):
        """Acquire semaphore."""
        self.semaphore.acquire()
        with self._lock:
            self.active_count += 1
            self.total_acquisitions += 1
            self.peak_concurrent = max(self.peak_concurrent, self.active_count)
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """Release semaphore."""
        with self._lock:
            self.active_count -= 1
        self.semaphore.release()
        return False
    
    def get_stats(self) -> dict:
        """Get semaphore statistics."""
        with self._lock:
            return {
                "active_connections": self.active_count,
                "max_concurrent": self.max_concurrent,
                "total_acquisitions": self.total_acquisitions,
                "peak_concurrent": self.peak_concurrent,
            }


# Initialize global rate limiter and concurrency semaphore
rate_limiter = SlidingWindowRateLimiter(
    max_requests=MAX_REQUESTS_PER_MINUTE,
    period_seconds=60,
    window_size=WINDOW_SIZE_SECONDS
)
concurrency_limiter = ConcurrencySemaphore(max_concurrent=SEARCH_WORKERS)

print(f"✅ Sliding window rate limiter initialized:")
print(f"   • {MAX_REQUESTS_PER_MINUTE} requests/min (8% safety margin)")
print(f"   • {WINDOW_SIZE_SECONDS}s windows to prevent bursts")
print(f"   • Max {rate_limiter.max_per_window} requests per {WINDOW_SIZE_SECONDS}s window")
print(f"✅ Concurrency limiter initialized: {SEARCH_WORKERS} simultaneous connections max")


✅ Sliding window rate limiter initialized:
   • 460 requests/min (8% safety margin)
   • 5s windows to prevent bursts
   • Max 38 requests per 5s window
✅ Concurrency limiter initialized: 10 simultaneous connections max


## 4️⃣ Entity Cache (CSV-based)

Persistent cache for entity IDs to avoid redundant API calls. The cache is stored in a CSV file and persists between notebook runs.

For production use, it can be in Redis or DB of choice. 


In [5]:
@dataclass
class EntityData:
    """Entity data structure for caching."""
    ticker: str
    entity_id: str
    company_name: str
    cached_at: str


class EntityCache:
    """
    CSV-based entity cache for persistent storage.
    Thread-safe for concurrent access.
    """
    
    def __init__(self, cache_file: Path):
        self.cache_file = cache_file
        self._cache: Dict[str, EntityData] = {}
        self._lock = threading.Lock()
        self._load_cache()
    
    def _load_cache(self) -> None:
        """Load existing cache from CSV file."""
        if not self.cache_file.exists():
            print(f"📝 Creating new entity cache: {self.cache_file}")
            return
        
        try:
            with open(self.cache_file, 'r', newline='', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    ticker = row['ticker'].upper()
                    self._cache[ticker] = EntityData(
                        ticker=ticker,
                        entity_id=row['entity_id'],
                        company_name=row['company_name'],
                        cached_at=row['cached_at']
                    )
            print(f"📂 Loaded {len(self._cache)} entities from cache")
        except Exception as e:
            print(f"⚠️ Error loading cache: {e}")
    
    def _save_cache(self) -> None:
        """Save cache to CSV file."""
        try:
            with open(self.cache_file, 'w', newline='', encoding='utf-8') as f:
                fieldnames = ['ticker', 'entity_id', 'company_name', 'cached_at']
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                for entity in self._cache.values():
                    writer.writerow(asdict(entity))
        except Exception as e:
            print(f"⚠️ Error saving cache: {e}")
    
    def get(self, ticker: str) -> Optional[EntityData]:
        """Get entity from cache."""
        with self._lock:
            return self._cache.get(ticker.upper())
    
    def set(self, ticker: str, entity_id: str, company_name: str) -> EntityData:
        """Add entity to cache and persist to file."""
        with self._lock:
            entity = EntityData(
                ticker=ticker.upper(),
                entity_id=entity_id,
                company_name=company_name,
                cached_at=datetime.now().isoformat()
            )
            self._cache[ticker.upper()] = entity
            self._save_cache()
            return entity
    
    def get_all(self) -> Dict[str, EntityData]:
        """Get all cached entities."""
        with self._lock:
            return dict(self._cache)
    
    def __len__(self) -> int:
        return len(self._cache)


# Initialize entity cache
entity_cache = EntityCache(ENTITY_CACHE_FILE)
print(f"✅ Entity cache initialized with {len(entity_cache)} entries")


📝 Creating new entity cache: output/entity_cache.csv
✅ Entity cache initialized with 0 entries


## 5️⃣ API Client Functions

Core functions for making API requests with rate limiting and retry logic.


In [6]:
def make_api_request(
    endpoint: str,
    payload: dict,
    max_retries: int = MAX_RETRIES,
    base_delay: float = 1.0
) -> Optional[dict]:
    """
    Make an API request with robust rate limiting, concurrency control, and automatic retry.
    
    Features:
    - Sliding window rate limiting (460 req/min with 5s windows)
    - Concurrency control (max 40 simultaneous connections)
    - Automatic retry up to 10,000 times with exponential backoff
    - Handles 429 (rate limit) and 403 (WAF) errors gracefully
    
    Args:
        endpoint: API endpoint path (e.g., '/search', '/knowledge-graph/companies')
        payload: Request JSON payload
        max_retries: Maximum retry attempts (default: 10,000)
        base_delay: Base delay for exponential backoff
        
    Returns:
        Response JSON or None if failed after all retries
    """
    url = f"{BASE_URL}{endpoint}"
    headers = {
        "X-API-KEY": API_KEY,
        "Content-Type": "application/json"
    }
    
    retry_count = 0
    warning_interval = 100  # Warn every 100 retries
    
    while retry_count < max_retries:
        # Acquire concurrency semaphore and rate limit token
        with concurrency_limiter:
            try:
                # Acquire rate limit permission (blocks if necessary)
                rate_limiter.acquire(timeout=120.0)
                
                response = http_session.post(
                    url,
                    headers=headers,
                    json=payload,
                    timeout=120
                )
                
                if response.status_code == 200:
                    return response.json()
                    
                elif response.status_code == 429:  # Rate limited by server
                    # Wait 1 second and retry automatically
                    retry_count += 1
                    if retry_count % warning_interval == 0:
                        rate_limiter.rate_limit_warnings += 1
                        print(f"⚠️ Rate limit warning: {retry_count} retries (429 errors)")
                    time.sleep(1.0)
                    continue
                    
                elif response.status_code == 403:  # Often WAF/Security block
                    # Wait 1 second and retry automatically with stronger backoff
                    retry_count += 1
                    if retry_count % warning_interval == 0:
                        rate_limiter.rate_limit_warnings += 1
                        print(f"🚨 Security filter warning: {retry_count} retries (403 errors)")
                    delay = min(base_delay * (1.5 ** (retry_count // 10)), 5.0)  # Cap at 5s
                    time.sleep(delay)
                    continue
                    
                else:
                    # Other errors - retry with exponential backoff
                    retry_count += 1
                    if retry_count < max_retries:
                        delay = min(base_delay * (2 ** min(retry_count, 10)), 30.0)  # Cap at 30s
                        if retry_count % warning_interval == 0:
                            print(f"⚠️ API error {response.status_code} after {retry_count} retries")
                        time.sleep(delay)
                        continue
                    else:
                        print(f"❌ Final API error {response.status_code}: {response.text[:200]}")
                        return None
                        
            except requests.exceptions.Timeout:
                retry_count += 1
                if retry_count % warning_interval == 0:
                    print(f"⏱️ Timeout after {retry_count} retries")
                if retry_count < max_retries:
                    time.sleep(base_delay * min(retry_count, 10))
                    continue
                return None
                
            except TimeoutError as e:
                # Rate limiter timeout
                print(f"⏰ Rate limiter timeout: {e}")
                retry_count += 1
                time.sleep(5.0)
                continue
                
            except Exception as e:
                retry_count += 1
                if retry_count % warning_interval == 0:
                    print(f"❌ Request error after {retry_count} retries: {e}")
                if retry_count < max_retries:
                    time.sleep(base_delay * min(retry_count, 10))
                    continue
                return None
    
    print(f"❌ Exhausted all {max_retries} retry attempts")
    return None


def lookup_entity(ticker: str) -> Optional[EntityData]:
    """
    Look up entity ID for a ticker using the Knowledge Graph API.
    Checks cache first, then calls API if not cached.
    
    Args:
        ticker: Stock ticker symbol
        
    Returns:
        EntityData or None if not found
    """
    ticker = ticker.upper()
    
    # Check cache first
    cached = entity_cache.get(ticker)
    if cached:
        return cached
    
    # Call Knowledge Graph API with automatic retry
    payload = {
        "query": ticker,
        "types": ["PUBLIC"]  # Only public companies
    }
    
    response = make_api_request("/knowledge-graph/companies", payload, max_retries=100)
    
    if response and response.get("results"):
        company = response["results"][0]
        entity = entity_cache.set(
            ticker=ticker,
            entity_id=company["id"],
            company_name=company["name"]
        )
        print(f"✅ Resolved: {ticker} → {company['name']} ({company['id']})")
        return entity
    else:
        print(f"❌ Could not resolve ticker: {ticker}")
        return None


print("✅ API client functions defined with robust retry logic")


✅ API client functions defined with robust retry logic


## 6️⃣ Parallel Entity Resolution

Resolve all tickers to entity IDs in parallel for large portfolios.


In [ ]:
def resolve_entities_parallel(tickers: List[str], max_workers: int = 10) -> Dict[str, EntityData]:
    """
    Resolve multiple tickers to entity IDs in parallel using ThreadPoolExecutor.
    
    Args:
        tickers: List of ticker symbols
        max_workers: Number of parallel workers
        
    Returns:
        Dictionary of ticker -> EntityData
    """
    print(f"\n🔍 Resolving {len(tickers)} tickers...")
    start_time = time.time()
    
    results = {}
    failed = []
    
    # Check which tickers are already cached
    tickers_to_lookup = []
    for ticker in tickers:
        cached = entity_cache.get(ticker)
        if cached:
            results[ticker] = cached
            print(f"  📦 Cache hit: {ticker} → {cached.company_name}")
        else:
            tickers_to_lookup.append(ticker)
    
    if not tickers_to_lookup:
        print(f"\n✅ All {len(results)} tickers resolved from cache")
        return results
    
    print(f"\n🌐 Looking up {len(tickers_to_lookup)} tickers via API...")
    
    # Parallel lookup for tickers not in cache
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_ticker = {
            executor.submit(lookup_entity, ticker): ticker
            for ticker in tickers_to_lookup
        }
        
        for future in as_completed(future_to_ticker):
            ticker = future_to_ticker[future]
            try:
                entity = future.result()
                if entity:
                    results[ticker] = entity
                else:
                    failed.append(ticker)
            except Exception as e:
                print(f"❌ Error resolving {ticker}: {e}")
                failed.append(ticker)
    
    elapsed = time.time() - start_time
    print(f"\n📊 Entity Resolution Summary:")
    print(f"   ✅ Resolved: {len(results)} tickers")
    print(f"   ❌ Failed: {len(failed)} tickers {failed if failed else ''}")
    print(f"   ⏱️ Time: {elapsed:.2f}s")
    
    return results


# Execute entity resolution
entities = resolve_entities_parallel(TICKERS, max_workers=ENTITY_WORKERS)

# Display resolved entities
print("\n📋 Resolved Entities:")
for ticker, entity in entities.items():
    print(f"   {ticker}: {entity.company_name} ({entity.entity_id})")


## 7️⃣ SQLite Database Setup

Create database tables for storing search results.


In [8]:
def setup_database(db_path: Path) -> sqlite3.Connection:
    """
    Create SQLite database and tables for storing search results.
    
    Args:
        db_path: Path to database file
        
    Returns:
        Database connection
    """
    conn = sqlite3.connect(str(db_path), check_same_thread=False)
    cursor = conn.cursor()
    
    # Create main search_results table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS search_results (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            search_id TEXT NOT NULL,
            ticker TEXT NOT NULL,
            company_name TEXT NOT NULL,
            entity_id TEXT NOT NULL,
            topic_name TEXT NOT NULL,
            topic_text TEXT NOT NULL,
            document_id TEXT NOT NULL,
            headline TEXT,
            timestamp TEXT,
            source_name TEXT,
            source_rank TEXT,
            document_url TEXT,
            document_type TEXT,
            chunk_text TEXT,
            chunk_relevance REAL,
            chunk_sentiment REAL,
            search_date TEXT NOT NULL,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP,
            UNIQUE(search_id, document_id, topic_name)
        )
    """)
    
    # Create indexes for faster queries
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_ticker ON search_results(ticker)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_topic ON search_results(topic_name)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_search_id ON search_results(search_id)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_timestamp ON search_results(timestamp)")
    
    # Create search_runs table to track search history
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS search_runs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            search_id TEXT UNIQUE NOT NULL,
            tickers TEXT NOT NULL,
            start_date TEXT NOT NULL,
            end_date TEXT NOT NULL,
            topics_count INTEGER,
            results_count INTEGER,
            duration_seconds REAL,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)
    
    conn.commit()
    return conn


# Initialize database
db_conn = setup_database(DATABASE_FILE)
print(f"✅ Database initialized: {DATABASE_FILE}")


✅ Database initialized: output/search_results.db


## 8️⃣ Search Functions

Core search functionality with parallel execution using ThreadPoolExecutor.


In [9]:
def format_timestamp(dt: datetime) -> str:
    """Format datetime to ISO 8601 with milliseconds and Z suffix."""
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    iso_str = dt.strftime('%Y-%m-%dT%H:%M:%S')
    milliseconds = dt.microsecond // 1000
    return f"{iso_str}.{milliseconds:03d}Z"


def search_topic(
    entity: EntityData,
    topic: dict,
    start_date: str,
    end_date: str
) -> List[dict]:
    """
    Search for a single ticker + topic combination.
    
    Args:
        entity: EntityData for the ticker
        topic: Topic dict with topic_name and topic_text
        start_date: Start date (YYYY-MM-DD)
        end_date: End date (YYYY-MM-DD)
        
    Returns:
        List of search result dictionaries
    """
    # Format topic text with company name
    topic_text = topic["topic_text"].format(company=entity.company_name)
    
    # Parse dates and convert to UTC
    start_dt = datetime.strptime(start_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    end_dt = datetime.strptime(end_date, "%Y-%m-%d").replace(
        hour=23, minute=59, second=59, tzinfo=timezone.utc
    )
    
    # Build search payload
    payload = {
        "query": {
            "text": topic_text,
            "filters": {
                "timestamp": {
                    "start": format_timestamp(start_dt),
                    "end": format_timestamp(end_dt)
                },
                "entity": {
                    "all_of": [entity.entity_id]
                },
                "document_type": {
                    "mode": "INCLUDE",
                    "values": DOCUMENT_TYPES
                },
                "sentiment": {
                    "values": SENTIMENT_VALUES
                }
            },
            "max_chunks": MAX_CHUNKS_PER_QUERY
        }
    }
    
    response = make_api_request("/search", payload, max_retries=5)
    
    results = []
    if response and response.get("results"):
        for doc in response["results"]:
            # Get best chunk (highest relevance)
            chunks = doc.get("chunks", [])
            if not chunks:
                continue
            
            best_chunk = max(chunks, key=lambda c: c.get("relevance", 0))
            
            results.append({
                "ticker": entity.ticker,
                "company_name": entity.company_name,
                "entity_id": entity.entity_id,
                "topic_name": topic["topic_name"],
                "topic_text": topic_text,
                "document_id": doc.get("id", ""),
                "headline": doc.get("headline", ""),
                "timestamp": doc.get("timestamp", ""),
                "source_name": doc.get("source", {}).get("name", ""),
                "source_rank": doc.get("source", {}).get("rank", ""),
                "document_url": doc.get("url", ""),
                "document_type": doc.get("document_type", "NEWS"),
                "chunk_text": best_chunk.get("text", ""),
                "chunk_relevance": best_chunk.get("relevance", 0),
                "chunk_sentiment": best_chunk.get("sentiment", 0),
            })
    
    return results


print("✅ Search functions defined")


✅ Search functions defined


In [10]:
def execute_parallel_search(
    entities: Dict[str, EntityData],
    topics: List[dict],
    start_date: str,
    end_date: str,
    max_workers: int = 40
) -> Tuple[str, List[dict], float]:
    """
    Execute search for all ticker+topic combinations in parallel with robust rate limiting.
    
    Features:
    - Sliding window rate limiting (460 req/min with 5s windows)
    - Concurrency control (max 40 simultaneous connections)
    - Automatic retry with up to 10,000 attempts
    
    Args:
        entities: Dictionary of ticker -> EntityData
        topics: List of topic dictionaries
        start_date: Start date (YYYY-MM-DD)
        end_date: End date (YYYY-MM-DD)
        max_workers: Number of parallel workers (also controls max simultaneous connections)
        
    Returns:
        Tuple of (search_id, list of all results, duration)
    """
    search_id = f"search_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    # Build all search task combinations
    search_tasks = []
    for ticker, entity in entities.items():
        for topic in topics:
            search_tasks.append((entity, topic))
    
    total_tasks = len(search_tasks)
    print(f"\n🚀 Starting parallel search with multi-layered protection")
    print(f"   📊 Search ID: {search_id}")
    print(f"   📈 Tickers: {len(entities)}")
    print(f"   📝 Topics: {len(topics)}")
    print(f"   🔢 Total queries: {total_tasks}")
    print(f"   👷 Workers: {max_workers}")
    print(f"   ⏱️ Rate limit: {MAX_REQUESTS_PER_MINUTE} req/min with {WINDOW_SIZE_SECONDS}s windows")
    print(f"   🔒 Concurrency limit: {SEARCH_WORKERS} simultaneous connections")
    print(f"   🔄 Auto-retry: Up to {MAX_RETRIES} attempts per request")
    print()
    
    start_time = time.time()
    all_results = []
    completed = 0
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_task = {
            executor.submit(
                search_topic, entity, topic, start_date, end_date
            ): (entity, topic)
            for entity, topic in search_tasks
        }
        
        for future in as_completed(future_to_task):
            entity, topic = future_to_task[future]
            completed += 1
            
            try:
                results = future.result()
                if results:
                    all_results.extend(results)
                    print(
                        f"   [{completed}/{total_tasks}] {entity.ticker} + {topic['topic_name']}: "
                        f"{len(results)} results"
                    )
                else:
                    print(
                        f"   [{completed}/{total_tasks}] {entity.ticker} + {topic['topic_name']}: "
                        f"0 results"
                    )
            except Exception as e:
                print(f"   ❌ [{completed}/{total_tasks}] {entity.ticker} + {topic['topic_name']}: Error - {e}")
    
    elapsed = time.time() - start_time
    
    # Get comprehensive stats from all three protection layers
    rate_stats = rate_limiter.get_stats()
    concurrency_stats = concurrency_limiter.get_stats()
    
    print(f"\n📊 Search Complete!")
    print(f"   ✅ Total results: {len(all_results)}")
    print(f"   ⏱️ Duration: {elapsed:.2f}s")
    print(f"   📈 Rate: {total_tasks / elapsed:.1f} queries/sec")
    
    print(f"\n🔧 Multi-layered Protection Stats:")
    print(f"   📊 Rate Limiter:")
    print(f"      • Total requests: {rate_stats['total_requests']}")
    print(f"      • Throttle events: {rate_stats['throttle_events']}")
    print(f"      • Total wait time: {rate_stats['total_wait_time_seconds']}s")
    print(f"      • Rate limit warnings: {rate_stats['rate_limit_warnings']}")
    print(f"      • Active requests in period: {rate_stats['current_requests_in_period']}/{rate_stats['max_requests_per_period']}")
    
    print(f"   🔒 Concurrency Limiter:")
    print(f"      • Total acquisitions: {concurrency_stats['total_acquisitions']}")
    print(f"      • Peak concurrent: {concurrency_stats['peak_concurrent']}/{concurrency_stats['max_concurrent']}")
    print(f"      • Currently active: {concurrency_stats['active_connections']}")
    
    return search_id, all_results, elapsed


print("✅ Parallel search executor defined with robust multi-layered protection")


✅ Parallel search executor defined with robust multi-layered protection


## 9️⃣ Execute Search & Store Results

Run the search and save results to SQLite database.


In [11]:
def save_results_to_db(
    conn: sqlite3.Connection,
    search_id: str,
    results: List[dict],
    tickers: List[str],
    start_date: str,
    end_date: str,
    topics_count: int,
    duration: float
) -> int:
    """
    Save search results to SQLite database.
    
    Args:
        conn: Database connection
        search_id: Unique search identifier
        results: List of result dictionaries
        tickers: List of tickers searched
        start_date: Search start date
        end_date: Search end date
        topics_count: Number of topics
        duration: Search duration in seconds
        
    Returns:
        Number of rows inserted
    """
    cursor = conn.cursor()
    
    # Insert search run metadata
    cursor.execute("""
        INSERT OR REPLACE INTO search_runs 
        (search_id, tickers, start_date, end_date, topics_count, results_count, duration_seconds)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (search_id, ",".join(tickers), start_date, end_date, topics_count, len(results), duration))
    
    # Insert results
    inserted = 0
    search_date = datetime.now().isoformat()
    
    for result in results:
        try:
            cursor.execute("""
                INSERT OR IGNORE INTO search_results (
                    search_id, ticker, company_name, entity_id, topic_name, topic_text,
                    document_id, headline, timestamp, source_name, source_rank,
                    document_url, document_type, chunk_text, chunk_relevance,
                    chunk_sentiment, search_date
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                search_id,
                result["ticker"],
                result["company_name"],
                result["entity_id"],
                result["topic_name"],
                result["topic_text"],
                result["document_id"],
                result["headline"],
                result["timestamp"],
                result["source_name"],
                result["source_rank"],
                result["document_url"],
                result["document_type"],
                result["chunk_text"],
                result["chunk_relevance"],
                result["chunk_sentiment"],
                search_date
            ))
            inserted += cursor.rowcount
        except Exception as e:
            print(f"⚠️ Error inserting result: {e}")
    
    conn.commit()
    return inserted


print("✅ Database save function defined")


✅ Database save function defined


In [ ]:
# =============================================================================
# 🚀 EXECUTE THE SEARCH
# =============================================================================

if entities:
    # Execute parallel search
    search_id, search_results, duration = execute_parallel_search(
        entities=entities,
        topics=TOPICS,
        start_date=START_DATE,
        end_date=END_DATE,
        max_workers=SEARCH_WORKERS
    )
    
    # Save results to database
    if search_results:
        rows_inserted = save_results_to_db(
            conn=db_conn,
            search_id=search_id,
            results=search_results,
            tickers=list(entities.keys()),
            start_date=START_DATE,
            end_date=END_DATE,
            topics_count=len(TOPICS),
            duration=duration
        )
        print(f"\n💾 Saved {rows_inserted} results to database")
        #print(f"📁 Database file: {DATABASE_FILE.absolute()}")
    else:
        print("\n⚠️ No results to save")
else:
    print("❌ No entities resolved. Cannot execute search.")


---

## 🔟 Query Results by Ticker

Retrieve and display results for a specific ticker from the database.


In [13]:
def query_by_ticker(conn: sqlite3.Connection, ticker: str, limit: int = 50) -> List[dict]:
    """
    Query search results for a specific ticker.
    
    Args:
        conn: Database connection
        ticker: Stock ticker symbol
        limit: Maximum results to return
        
    Returns:
        List of result dictionaries
    """
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT 
            ticker, company_name, topic_name, headline, timestamp,
            source_name, chunk_relevance, chunk_sentiment, chunk_text, document_url
        FROM search_results
        WHERE ticker = ?
        ORDER BY timestamp DESC, chunk_relevance DESC
        LIMIT ?
    """, (ticker.upper(), limit))
    
    columns = [desc[0] for desc in cursor.description]
    results = [dict(zip(columns, row)) for row in cursor.fetchall()]
    
    return results


# =============================================================================
# 🔍 QUERY BY TICKER - Change the ticker below to query different stocks
# =============================================================================
QUERY_TICKER = "AAPL"  # <-- MODIFY THIS

ticker_results = query_by_ticker(db_conn, QUERY_TICKER, limit=20)

print(f"\n📊 Results for {QUERY_TICKER}: {len(ticker_results)} articles")
print("=" * 80)

for i, result in enumerate(ticker_results[:10], 1):
    sentiment_emoji = "🟢" if result['chunk_sentiment'] and result['chunk_sentiment'] > 0 else "🔴" if result['chunk_sentiment'] and result['chunk_sentiment'] < 0 else "⚪"
    print(f"\n{i}. [{result['topic_name']}] {sentiment_emoji}")
    headline = result.get('headline', 'No headline')
    print(f"   📰 {headline[:80]}{'...' if len(headline) > 80 else ''}")
    print(f"   📅 {result['timestamp']} | 📡 {result['source_name']}")
    sentiment_str = f"{result['chunk_sentiment']:.2f}" if result['chunk_sentiment'] else "N/A"
    relevance_str = f"{result['chunk_relevance']:.3f}" if result['chunk_relevance'] else "N/A"
    print(f"   📈 Relevance: {relevance_str} | Sentiment: {sentiment_str}")
    chunk = result.get('chunk_text', '')
    print(f"   📝 {chunk[:150]}...")



📊 Results for AAPL: 20 articles

1. [Regulatory] 🔴
   📰 UK court begins hearing $1.83 billion lawsuit against Apple over App Store fees
   📅 2025-01-12T23:27:11 | 📡 SiliconANGLE
   📈 Relevance: 0.862 | Sentiment: -0.85
   📝 Lawyers representing Apple Inc. will be in court Monday in the U.K. to face a class action lawsuit that alleges that the iPhone maker runs its App Sto...

2. [Financing] 🟢
   📰 These American Companies Are Rolling Back Some DEI Policies
   📅 2025-01-12T21:38:03 | 📡 Epoch Times
   📈 Relevance: 0.339 | Sentiment: 0.21
   📝 Apple's board also responded to a proposal from the National Center for Public Policy by urging shareholders to keep intact its DEI mandates. "Apple i...

3. [Financing] 🟢
   📰 Apple Is the Largest Company in the World. Here's Why Investors Should Be Wary o...
   📅 2025-01-12T21:21:09 | 📡 Yahoo! Finance
   📈 Relevance: 0.643 | Sentiment: 0.54
   📝 Apple has done a lot of repurchasing of its own shares, so that its total yield to shareholders, inclu

## 1️⃣1️⃣ Query Results by Topic

Retrieve and display results for a specific topic across all tickers.


In [14]:
def query_by_topic(conn: sqlite3.Connection, topic_name: str, limit: int = 50) -> List[dict]:
    """
    Query search results for a specific topic across all tickers.
    
    Args:
        conn: Database connection
        topic_name: Topic name to search
        limit: Maximum results to return
        
    Returns:
        List of result dictionaries
    """
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT 
            ticker, company_name, topic_name, headline, timestamp,
            source_name, chunk_relevance, chunk_sentiment, chunk_text, document_url
        FROM search_results
        WHERE topic_name = ?
        ORDER BY timestamp DESC, chunk_relevance DESC
        LIMIT ?
    """, (topic_name, limit))
    
    columns = [desc[0] for desc in cursor.description]
    results = [dict(zip(columns, row)) for row in cursor.fetchall()]
    
    return results


def get_available_topics(conn: sqlite3.Connection) -> List[str]:
    """Get list of all topics in the database."""
    cursor = conn.cursor()
    cursor.execute("SELECT DISTINCT topic_name FROM search_results ORDER BY topic_name")
    return [row[0] for row in cursor.fetchall()]


# Show available topics
available_topics = get_available_topics(db_conn)
print("📋 Available topics in database:")
for topic in available_topics:
    print(f"   • {topic}")


📋 Available topics in database:
   • Competition
   • Costs
   • Financial Metrics
   • Financing
   • Industry
   • Leadership
   • M&A
   • Markets
   • Products
   • Regulatory
   • Supply Chain


In [15]:
# =============================================================================
# 🔍 QUERY BY TOPIC - Change the topic below to query different topics
# =============================================================================
QUERY_TOPIC = "Leadership"  # <-- MODIFY THIS
# Initialize database

topic_results = query_by_topic(db_conn, QUERY_TOPIC, limit=30)

print(f"\n📊 Results for topic '{QUERY_TOPIC}': {len(topic_results)} articles")
print("=" * 80)

# Group results by ticker for better readability
grouped = defaultdict(list)
for result in topic_results:
    grouped[result['ticker']].append(result)

for ticker, results in grouped.items():
    print(f"\n🏢 {ticker} ({results[0]['company_name']})")
    print("-" * 60)
    
    for result in results[:5]:  # Show top 5 per ticker
        sentiment_emoji = "🟢" if result['chunk_sentiment'] and result['chunk_sentiment'] > 0 else "🔴" if result['chunk_sentiment'] and result['chunk_sentiment'] < 0 else "⚪"
        headline = result.get('headline', 'No headline')
        print(f"   {sentiment_emoji} {headline[:70]}{'...' if len(headline) > 70 else ''}")
        relevance_str = f"{result['chunk_relevance']:.3f}" if result['chunk_relevance'] else "N/A"
        print(f"      📅 {result['timestamp']} | 📈 Rel: {relevance_str}")



📊 Results for topic 'Leadership': 30 articles

🏢 TSLA (Tesla Inc.)
------------------------------------------------------------
   🟢 Elon Musk could become a 'special government employee' as a co-lead of...
      📅 2025-01-12T23:25:06 | 📈 Rel: 0.775
   🟢 Self-driving-car executives excited for Trump (and Musk) to take the w...
      📅 2025-01-11T11:04:19 | 📈 Rel: 0.424

🏢 CEG (Constellation Energy Corp.)
------------------------------------------------------------
   🟢 Here's Why Constellation Energy Group (CEG) Led This Week's Rally
      📅 2025-01-12T22:23:08 | 📈 Rel: 0.587

🏢 VRTX (Vertex Pharmaceuticals Inc.)
------------------------------------------------------------
   🟢 Vertex Provides Pipeline and Business Updates in Advance of Upcoming I...
      📅 2025-01-12T22:07:12 | 📈 Rel: 0.312

🏢 NVDA (NVIDIA Corp.)
------------------------------------------------------------
   🟢 AI comes down from the cloud as chips get smarter
      📅 2025-01-12T21:41:11 | 📈 Rel: 0.768

🏢 PLTR (Pala

## 1️⃣2️⃣ Summary Statistics

View aggregated statistics from the database.


In [16]:
def get_database_stats(conn: sqlite3.Connection) -> dict:
    """Get summary statistics from the database."""
    cursor = conn.cursor()
    
    # Total results
    cursor.execute("SELECT COUNT(*) FROM search_results")
    total_results = cursor.fetchone()[0]
    
    # Results by ticker
    cursor.execute("""
        SELECT ticker, company_name, COUNT(*) as count 
        FROM search_results 
        GROUP BY ticker 
        ORDER BY count DESC
    """)
    by_ticker = cursor.fetchall()
    
    # Results by topic
    cursor.execute("""
        SELECT topic_name, COUNT(*) as count 
        FROM search_results 
        GROUP BY topic_name 
        ORDER BY count DESC
    """)
    by_topic = cursor.fetchall()
    
    # Average sentiment by ticker
    cursor.execute("""
        SELECT ticker, AVG(chunk_sentiment) as avg_sentiment
        FROM search_results
        GROUP BY ticker
    """)
    sentiment_by_ticker = cursor.fetchall()
    
    return {
        "total_results": total_results,
        "by_ticker": by_ticker,
        "by_topic": by_topic,
        "sentiment_by_ticker": sentiment_by_ticker,
    }


# Display statistics
stats = get_database_stats(db_conn)

print("\n" + "=" * 80)
print("📊 DATABASE STATISTICS")
print("=" * 80)

print(f"\n📈 Total Results: {stats['total_results']}")

print(f"\n📊 Results by Ticker:")
for ticker, company, count in stats['by_ticker']:
    company_short = company[:25] if company else "Unknown"
    print(f"   {ticker:6} ({company_short:25}): {count:4} articles")

print(f"\n📝 Results by Topic:")
for topic, count in stats['by_topic']:
    print(f"   {topic:15}: {count:4} articles")

print(f"\n🎭 Average Sentiment by Ticker:")
for ticker, sentiment in stats['sentiment_by_ticker']:
    if sentiment:
        emoji = "🟢" if sentiment > 0.1 else "🔴" if sentiment < -0.1 else "⚪"
        print(f"   {ticker:6}: {emoji} {sentiment:.3f}")
    else:
        print(f"   {ticker:6}: ⚪ N/A")



📊 DATABASE STATISTICS

📈 Total Results: 7298

📊 Results by Ticker:
   TSLA   (Tesla Inc.               ):  179 articles
   AAPL   (Apple Inc.               ):  157 articles
   MSFT   (Microsoft Corp.          ):  154 articles
   AMZN   (Amazon.com Inc.          ):  153 articles
   NVDA   (NVIDIA Corp.             ):  153 articles
   META   (Meta Platforms Inc.      ):  152 articles
   PLTR   (Palantir Technologies Inc):  127 articles
   MU     (Micron Technology Inc.   ):  124 articles
   CEG    (Constellation Energy Corp):  123 articles
   AMD    (Advanced Micro Devices In):  117 articles
   REGN   (Regeneron Pharmaceuticals):  114 articles
   WBD    (Warner Bros Discovery Inc):  112 articles
   ASML   (ASML Holding N.V.        ):  111 articles
   INTC   (Intel Corp.              ):  110 articles
   AEP    (American Electric Power C):  108 articles
   COST   (Costco Wholesale Corp.   ):  108 articles
   GOOGL  (Alphabet Inc.            ):  105 articles
   GILD   (Gilead Sciences Inc.

## 1️⃣3️⃣ Cleanup & Summary


In [17]:
# Close database connection
db_conn.close()
print("✅ Database connection closed")
http_session.close()


# Final summary
print("\n" + "=" * 80)
print("🎉 PORTFOLIO NEWS SEARCH COMPLETE!")
print("=" * 80)
#print(f"\n📁 Output files saved to: {OUTPUT_DIR.absolute()}")
print(f"   • Entity cache: {ENTITY_CACHE_FILE.name}")
print(f"   • Database: {DATABASE_FILE.name}")

# Display comprehensive multi-layered protection stats
print("\n" + "=" * 80)
print("🔧 MULTI-LAYERED PROTECTION FINAL STATS")
print("=" * 80)

rate_stats = rate_limiter.get_stats()
concurrency_stats = concurrency_limiter.get_stats()

print("\n🛡️ Layer 1: Sliding Window Rate Limiter")
print(f"   • Total API requests made: {rate_stats['total_requests']:,}")
print(f"   • Throttle events (brief pauses): {rate_stats['throttle_events']:,}")
print(f"   • Total wait time: {rate_stats['total_wait_time_seconds']:.2f}s ({rate_stats['total_wait_time_seconds']/60:.1f} min)")
print(f"      └─ Note: Happens in parallel across {SEARCH_WORKERS} threads, not sequentially")
print(f"   • Rate limit warnings (429/403): {rate_stats['rate_limit_warnings']:,}")
print(f"   • Current window: {rate_stats['current_requests_in_period']}/{rate_stats['max_requests_per_period']} requests")

print("\n🔒 Layer 2: Concurrency Semaphore")
print(f"   • Total connection acquisitions: {concurrency_stats['total_acquisitions']:,}")
print(f"   • Peak concurrent connections: {concurrency_stats['peak_concurrent']}/{concurrency_stats['max_concurrent']}")
print(f"   • Currently active connections: {concurrency_stats['active_connections']}")

print("\n🔄 Layer 3: Automatic Retry")
print(f"   • Max retry attempts configured: {MAX_RETRIES}")
print(f"   • Retry strategy: 1s wait on 429, progressive backoff on 403")
print(f"   • Total warnings issued: {rate_stats['rate_limit_warnings']:,}")

print("\n✅ All protection layers worked successfully to ensure safe API usage!")


✅ Database connection closed

🎉 PORTFOLIO NEWS SEARCH COMPLETE!
   • Entity cache: entity_cache.csv
   • Database: search_results.db

🔧 MULTI-LAYERED PROTECTION FINAL STATS

🛡️ Layer 1: Sliding Window Rate Limiter
   • Total API requests made: 2,620
   • Throttle events (brief pauses): 920
   • Total wait time: 460.00s (7.7 min)
      └─ Note: Happens in parallel across 10 threads, not sequentially
   • Rate limit warnings (429/403): 0
   • Current window: 395/460 requests

🔒 Layer 2: Concurrency Semaphore
   • Total connection acquisitions: 2,620
   • Peak concurrent connections: 10/10
   • Currently active connections: 0

🔄 Layer 3: Automatic Retry
   • Max retry attempts configured: 5
   • Retry strategy: 1s wait on 429, progressive backoff on 403
   • Total warnings issued: 0

✅ All protection layers worked successfully to ensure safe API usage!


---

## 📖 How to Customize This Notebook

### Adding/Modifying Tickers

Edit the `TICKERS_INPUT` variable in Cell 2:
```python
TICKERS_INPUT = "AAPL, MSFT, GOOGL, META, AMZN, NVDA, TSLA"
```

### Changing Date Range

Edit `START_DATE` and `END_DATE` in Cell 2:
```python
START_DATE = "2024-12-01"
END_DATE = "2024-12-31"
```

### Adding Custom Topics

Add topics to the `TOPICS` list in Cell 2:
```python
TOPICS = [
    {"topic_name": "AI Strategy", "topic_text": "What AI initiatives has {company} announced?"},
    {"topic_name": "Supply Chain", "topic_text": "What supply chain issues affect {company}?"},
    # Add more topics...
]
```

---

## 📚 API Documentation
- [Bigdata.com API Docs](https://docs.bigdata.com)
- [Knowledge Graph API](https://docs.bigdata.com/getting-started/knowledge_graph)
- [Search API](https://docs.bigdata.com/getting-started/search)

---

